In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *
from utils_kga.statistical_analysis.get_spin_and_bond_angle_statistics import get_bond_angle_interval_ion_pair_statistics

In [2]:
# Load edge-df
with open("data/dfs_of_magnetic_edge_information.json") as f:
# replace to analyze all structs, not only cryst. uniques:
#with open("data/dfs_of_magnetic_edge_information_include_crystallographic_multiples.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("../../data_retrieval_and_preprocessing_MAGNDATA/data/df_grouped_and_chosen_commensurate_MAGNDATA.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "plots/TM_octahedra_ion_pair_distributions_at_90_bond_angles"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
# Add is_tm bool for later easier analysis
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))

    ang_df["ion_pair"] = ang_df.apply(lambda row: (row["site_element"],
                    (int(row["site_oxidation"])),
                    row["site_to_element"],
                    int(row["site_to_oxidation"])), axis=1)

In [4]:
# (80, 100) bond angle interval
custom_sort_by_electron_configuration = [
    ("V", 3), ("Re", 5),
    ("Cr", 3), ("Mn", 4), ("Ru", 5),
    ("Cr", 2), ("Mn", 3),
    ("Mn", 2), ("Fe", 3),
    ("Ru", 3), ("Ir", 4),
    ("Mn", 1),
    ("Fe", 2), ("Co", 3),
    ("Co", 2),
    ("Ni", 2),
    ("Cu", 2)
]

In [5]:
write_mode = "w"
bond_angle_range90 = (80.0, 100.0)
normalize_bool = False
normalize_string = "absolute occurrences"
data_string = f"{bond_angle_range90[0]}°-{bond_angle_range90[1]}° bond angles with TM octahedra at both nodes"

occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    n_lattice_points = df.at[md_id, "n_lattice_points"]

    for mag_type, condition in zip(["afm", "fm", ],
                                    [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
        type_df = subdf.loc[condition]
        if not type_df.empty:
            abs_ion_pairs = get_bond_angle_interval_ion_pair_statistics(df=type_df, include_ligand_multiplicity=False,
                                                        analyze_column="ion_pair", n_lattice_points=n_lattice_points,
                                                        bond_angle_interval=bond_angle_range90)

            for k, v in abs_ion_pairs.items():
                # Assert no detected ions are missing
                try:
                    assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                    assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                except AssertionError as e:
                    print(str(e), k)

                if k in occus[mag_type]:
                    occus[mag_type][k] += v
                else:
                    occus[mag_type][k] = v

                if k in num_structures:
                    num_structures[k].add(md_id)
                else:
                    num_structures[k] = {md_id}

print(occus)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
ratio_fig.write_image(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))

print("--------------- num structures ------------------------")
print(num_structures)
n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, 
                                         ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
numstructfig.write_image(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))

{'afm': {('Co', 2, 'Co', 2): 136.0, ('Mn', 4, 'Mn', 4): 88.0, ('Fe', 3, 'Fe', 3): 204.00000000000009, ('Mn', 2, 'Mn', 2): 346.0, ('Ni', 2, 'Ni', 2): 78.0, ('Fe', 2, 'Fe', 2): 90.66666666666666, ('Fe', 2, 'Fe', 3): 40.0, ('Fe', 3, 'Fe', 2): 40.0, ('Cr', 3, 'Cr', 3): 170.66666666666663, ('Ir', 4, 'Ir', 4): 3.9999999999999982, ('Ru', 5, 'Ru', 5): 11.999999999999996, ('V', 3, 'V', 3): 136.0, ('Cr', 2, 'Cr', 2): 10.0, ('Mn', 3, 'Mn', 3): 108.0, ('Co', 3, 'Co', 3): 4.0, ('Cu', 2, 'Cu', 2): 48.0, ('Re', 5, 'Re', 5): 8.0, ('Ru', 3, 'Ru', 3): 20.0}, 'fm': {('Mn', 2, 'Mn', 2): 204.0, ('Co', 2, 'Co', 2): 127.99999999999999, ('Co', 3, 'Co', 3): 15.999999999999996, ('Fe', 3, 'Cu', 2): 15.999999999999993, ('Cu', 2, 'Fe', 3): 15.999999999999993, ('Cu', 2, 'Cu', 2): 36.0, ('Fe', 3, 'Ni', 2): 10.666666666666664, ('Ni', 2, 'Fe', 3): 10.666666666666664, ('Ni', 2, 'Ni', 2): 195.99999999999991, ('Fe', 2, 'Fe', 2): 154.66666666666663, ('Fe', 2, 'Fe', 3): 100.0, ('Fe', 3, 'Fe', 2): 100.0, ('Fe', 3, 'Fe', 3):

In [6]:
# Repeat analysis considering only oxygen edges
write_mode = "w"
bond_angle_range90 = (80.0, 100.0)
normalize_bool = False
normalize_string = "absolute occurrences"
data_string = f"{bond_angle_range90[0]}°-{bond_angle_range90[1]}° bond angles, oxygen-conn TM octahedra"

occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

    n_lattice_points = df.at[md_id, "n_lattice_points"]

    for mag_type, condition in zip(["afm", "fm", ],
                                    [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
        type_df = subdf.loc[condition]
        if not type_df.empty:
            abs_ion_pairs = get_bond_angle_interval_ion_pair_statistics(df=type_df, include_ligand_multiplicity=False,
                                                        analyze_column="ion_pair", n_lattice_points=n_lattice_points,
                                                        bond_angle_interval=bond_angle_range90)

            for k, v in abs_ion_pairs.items():
                # Assert no detected ions are missing
                assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))


                if k in occus[mag_type]:
                    occus[mag_type][k] += v
                else:
                    occus[mag_type][k] = v

                if k in num_structures:
                    num_structures[k].add(md_id)
                else:
                    num_structures[k] = {md_id}

print(occus)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)

ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
#ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
ratio_fig.write_image(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))


close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))

print("--------------- num structures ------------------------")
print(num_structures)
n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, 
                                         ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
numstructfig.write_image(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))

{'afm': {('Co', 2, 'Co', 2): 132.0, ('Mn', 4, 'Mn', 4): 88.0, ('Fe', 3, 'Fe', 3): 204.00000000000009, ('Mn', 2, 'Mn', 2): 286.0, ('Ni', 2, 'Ni', 2): 74.0, ('Fe', 2, 'Fe', 2): 60.0, ('Fe', 2, 'Fe', 3): 40.0, ('Fe', 3, 'Fe', 2): 40.0, ('Cr', 3, 'Cr', 3): 75.99999999999997, ('Ir', 4, 'Ir', 4): 3.9999999999999982, ('Ru', 5, 'Ru', 5): 11.999999999999996, ('V', 3, 'V', 3): 124.0, ('Cr', 2, 'Cr', 2): 10.0, ('Mn', 3, 'Mn', 3): 108.0, ('Co', 3, 'Co', 3): 4.0, ('Cu', 2, 'Cu', 2): 48.0, ('Re', 5, 'Re', 5): 8.0}, 'fm': {('Mn', 2, 'Mn', 2): 122.0, ('Co', 2, 'Co', 2): 92.0, ('Co', 3, 'Co', 3): 15.999999999999996, ('Fe', 3, 'Cu', 2): 15.999999999999993, ('Cu', 2, 'Fe', 3): 15.999999999999993, ('Cu', 2, 'Cu', 2): 20.0, ('Fe', 3, 'Ni', 2): 10.666666666666664, ('Ni', 2, 'Fe', 3): 10.666666666666664, ('Ni', 2, 'Ni', 2): 151.99999999999994, ('Fe', 2, 'Fe', 2): 84.0, ('Fe', 2, 'Fe', 3): 100.0, ('Fe', 3, 'Fe', 2): 100.0, ('Fe', 3, 'Fe', 3): 296.00000000000006, ('Mn', 3, 'Mn', 3): 60.0, ('Ir', 4, 'Ir', 4): 4